# Module 4 - Class 1: Linear Regression

Linear, multiple, polynomial regression, and a gradient descent demo.


## 0. Setup


In [ ]:
# Import the libraries used in this notebook.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")


## 1. Load Data


In [ ]:
# Superstore dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/Superstore.csv"

try:
    df = pd.read_csv(url, encoding='windows-1252')
except Exception:
    df = pd.read_csv(url)

print(f"Shape: {df.shape}")
df.head()


In [ ]:
# Fallback: upload if URL fails
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0], encoding='windows-1252')


In [ ]:
# Quick inspection of numeric columns
df[['Sales', 'Quantity', 'Discount', 'Profit']].describe()


## 2. Simple Linear Regression: Sales ~ Discount


In [ ]:
# Scatter plot to see the relationship
plt.figure(figsize=(8, 5))
plt.scatter(df['Discount'], df['Sales'], alpha=0.3, s=10)
plt.xlabel('Discount')
plt.ylabel('Sales')
plt.title('Sales vs Discount')
plt.show()


In [ ]:
# Prepare data
X_simple = df[['Discount']]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

# Train simple linear regression
lr_simple = LinearRegression()
lr_simple.fit(X_train, y_train)
y_pred_simple = lr_simple.predict(X_test)

print(f"Coefficient (slope): {lr_simple.coef_[0]:.4f}")
print(f"Intercept: {lr_simple.intercept_:.4f}")
print(f"\nInterpretation: For each 1-unit increase in Discount,")
print(f"Sales changes by {lr_simple.coef_[0]:.2f} on average.")


## 3. Multiple Linear Regression


In [ ]:
# Use multiple features
X_multi = df[['Discount', 'Quantity', 'Profit']]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

lr_multi = LinearRegression()
lr_multi.fit(X_train_m, y_train_m)
y_pred_multi = lr_multi.predict(X_test_m)

# Coefficients
coef_df = pd.DataFrame({
    'Feature': X_multi.columns,
    'Coefficient': lr_multi.coef_
})
print("Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {lr_multi.intercept_:.4f}")


## 4. Evaluation Metrics


In [ ]:
# Evaluate regression performance.
def evaluate_regression(y_true, y_pred, model_name="Model"):
    """Compute and print regression metrics."""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"--- {model_name} ---")
    print(f"MSE:  {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE:  {mae:.2f}")
    print(f"R2:   {r2:.4f}")
    print()
    return {'model': model_name, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

metrics_simple = evaluate_regression(y_test, y_pred_simple, "Simple LR (Discount only)")
metrics_multi = evaluate_regression(y_test_m, y_pred_multi, "Multiple LR (Discount + Quantity + Profit)")


## 5. Coefficient Interpretation


In [ ]:
# Visualize coefficients
plt.figure(figsize=(8, 4))
plt.barh(coef_df['Feature'], coef_df['Coefficient'])
plt.xlabel('Coefficient Value')
plt.title('Multiple Linear Regression Coefficients')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

print("Each coefficient tells you: holding other features constant,")
print("a 1-unit increase in this feature changes Sales by this amount.")


## 6. Residual Plot


In [ ]:
# Visualize prediction residuals.
residuals = y_test_m - y_pred_multi

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_pred_multi, residuals, alpha=0.3, s=10)
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Sales')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted Values')

# Residual distribution
axes[1].hist(residuals, bins=50, edgecolor='black')
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print("Ideal residual plot: randomly scattered around 0, no pattern.")
print("A pattern suggests the model is missing something (non-linearity, interactions, etc.).")


In [ ]:
# Train polynomial regression models with degree 2 and degree 3 features.
poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly2 = poly2.fit_transform(X_train_m)
X_test_poly2 = poly2.transform(X_test_m)

lr_poly2 = LinearRegression()
lr_poly2.fit(X_train_poly2, y_train_m)
y_pred_poly2 = lr_poly2.predict(X_test_poly2)
metrics_poly2 = evaluate_regression(y_test_m, y_pred_poly2, "Polynomial LR (degree 2)")

poly3 = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly3 = poly3.fit_transform(X_train_m)
X_test_poly3 = poly3.transform(X_test_m)

lr_poly3 = LinearRegression()
lr_poly3.fit(X_train_poly3, y_train_m)
y_pred_poly3 = lr_poly3.predict(X_test_poly3)
metrics_poly3 = evaluate_regression(y_test_m, y_pred_poly3, "Polynomial LR (degree 3)")


In [ ]:
# Compare all regression models in one metrics table.
comparison_rows = [metrics_simple, metrics_multi, metrics_poly2, metrics_poly3]
comparison_df = (
    pd.DataFrame(comparison_rows)
    .rename(columns={"model": "Model"})
    [["Model", "MSE", "RMSE", "MAE", "R2"]]
    .round(4)
)

print(comparison_df.to_string(index=False))


## 8. Gradient Descent Visualization (Demo)

This cell shows how gradient descent minimizes the loss function step by step. You do not need to modify this -- just run it and observe.


In [ ]:
# Simple gradient descent demo on a 1D problem
# Minimize MSE for: y = w * x (single weight, no bias)

np.random.seed(42)
x_gd = np.random.randn(100)
y_gd = 3.0 * x_gd + np.random.randn(100) * 0.5  # true weight = 3.0

# Gradient descent
w = 0.0  # initial weight
lr = 0.05  # learning rate
n_epochs = 50
history = []

for epoch in range(n_epochs):
    # Prediction
    y_hat = w * x_gd
    # Loss (MSE)
    loss = np.mean((y_gd - y_hat) ** 2)
    # Gradient: d(MSE)/dw = -2 * mean(x * (y - y_hat))
    grad = -2 * np.mean(x_gd * (y_gd - y_hat))
    # Update
    w = w - lr * grad
    history.append({'epoch': epoch, 'weight': w, 'loss': loss})

history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss over epochs
axes[0].plot(history_df['epoch'], history_df['loss'], 'b-o', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss Decreasing Over Epochs')

# Weight convergence
axes[1].plot(history_df['epoch'], history_df['weight'], 'r-o', markersize=3)
axes[1].axhline(y=3.0, color='green', linestyle='--', label='True weight (3.0)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Weight Value')
axes[1].set_title('Weight Converging to True Value')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Final weight: {w:.4f} (true: 3.0)")
print(f"Final loss: {history_df['loss'].iloc[-1]:.4f}")
